# 06 - Batch Reprocessing (Lambda Architecture)
Retraitement periodique des donnees streaming accumulees dans HDFS

**Principe** : les donnees arrivent en streaming (notebook 04/05) et sont stockees dans HDFS.
A intervalle regulier (configurable), ce notebook lance un traitement batch complet
sur toutes les donnees accumulees pour produire des resultats plus precis.

```
Streaming (temps reel)          Batch Reprocessing (periodique)
   ├─ approximatif               ├─ exact et complet
   ├─ latence faible             ├─ latence = intervalle
   └─ ClickHouse stream_*        └─ ClickHouse reprocess_*
```

In [1]:
import subprocess
subprocess.check_call(['pip', 'install', 'clickhouse-connect'])

import builtins
import time
import datetime
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
import clickhouse_connect

spark = SparkSession.builder \
    .appName('06_Batch_Reprocessing') \
    .master('spark://spark-master:7077') \
    .config('spark.hadoop.fs.defaultFS', 'hdfs://namenode:9000') \
    .config('spark.sql.shuffle.partitions', '6') \
    .getOrCreate()

ch_client = clickhouse_connect.get_client(host='clickhouse', port=8123)

# Charger les dimensions (statiques)
locations = spark.read.parquet('hdfs://namenode:9000/projet/silver/locations_clean/')
dates = spark.read.parquet('hdfs://namenode:9000/projet/silver/dates_clean/')

print('Spark connecte')
print('ClickHouse connecte')
print(f'Locations: {locations.count()} zones')

Spark connecte
ClickHouse connecte
Locations: 263 zones


In [ ]:
# Garantir que les tables reprocess_* existent (idempotent)
import clickhouse_connect as _ch
_c = _ch.get_client(host='clickhouse', port=8123)
_c.command("""
    CREATE TABLE IF NOT EXISTS reprocess_borough_stats (
        borough String, nb_trips UInt64, total_revenue Float64,
        avg_distance Float64, avg_fare Float64
    ) ENGINE = ReplacingMergeTree() ORDER BY borough
""")
_c.command("""
    CREATE TABLE IF NOT EXISTS reprocess_top_zones (
        zone String, borough String, nb_trips UInt64
    ) ENGINE = ReplacingMergeTree() ORDER BY (zone, borough)
""")
_c.command("""
    CREATE TABLE IF NOT EXISTS reprocess_daily_stats (
        day UInt8, nb_trips UInt64, total_revenue Float64, avg_fare Float64
    ) ENGINE = ReplacingMergeTree() ORDER BY day
""")
_c.command("""
    CREATE TABLE IF NOT EXISTS reprocess_log (
        cycle_num UInt32, executed_at DateTime, rows_processed UInt64,
        duration_seconds Float64, boroughs_found UInt8, zones_found UInt8
    ) ENGINE = MergeTree() ORDER BY cycle_num
""")
_c.close()
print('Tables reprocess_* OK')

In [ ]:
# ══════════════════════════════════════════════════
# BATCH WATCHER - Lance en arriere-plan depuis Jupyter
# Herite de l'environnement complet (DNS, HADOOP, JAVA)
# Le bouton du dashboard ecrit dans batch_trigger → ce
# watcher le detecte et lance le reprocessing Spark
# ══════════════════════════════════════════════════
import subprocess, sys, os, signal

def start_watcher():
    log = open('/tmp/batch_watcher.log', 'w')
    proc = subprocess.Popen(
        [sys.executable, '/home/jovyan/work/scripts/batch_watcher.py'],
        stdout=log,
        stderr=log,
        env=os.environ.copy()   # herite de tout l'env Jupyter
    )
    print(f'Batch Watcher demarre (PID {proc.pid})')
    print(f'Logs : /tmp/batch_watcher.log')
    return proc

def stop_watcher(proc):
    if proc and proc.poll() is None:
        proc.send_signal(signal.SIGTERM)
        proc.wait(timeout=5)
        print('Batch Watcher arrete')

# Tuer un ancien watcher s'il tourne encore (docker exec ou precedente cellule)
subprocess.run(['pkill', '-f', 'batch_watcher.py'], capture_output=True)

watcher_proc = start_watcher()

In [ ]:
# ══════════════════════════════════════════════════
# CONFIGURATION
# ══════════════════════════════════════════════════

HDFS_STREAMING_PATH = 'hdfs://namenode:9000/projet/streaming/december_trips'
HDFS_CHECKPOINT_PATH = 'hdfs://namenode:9000/projet/streaming/_checkpoint_december'
SILVER_TRIPS_PATH = 'hdfs://namenode:9000/projet/silver/trips_clean/'
KAFKA_BROKER = 'kafka:29092'
KAFKA_TOPIC = 'nyc-taxi-trips'

# Intervalle entre chaque batch reprocessing (en secondes)
BATCH_INTERVAL_SECONDS = 30

# Nombre de cycles de reprocessing a executer
# Mettre -1 pour tourner indefiniment (arreter manuellement)
NB_CYCLES = 5

print(f'Intervalle : {BATCH_INTERVAL_SECONDS} secondes')
print(f'Cycles prevus : {NB_CYCLES}')
print(f'Source HDFS : {HDFS_STREAMING_PATH}')
print(f'Silver dest : {SILVER_TRIPS_PATH}')

In [ ]:
def run_batch_reprocessing(cycle_num):
    """Execute un cycle de batch reprocessing sur les donnees streaming accumulees"""
    
    start = datetime.datetime.now()
    print(f'\n{"="*60}')
    print(f'CYCLE {cycle_num} - {start.strftime("%H:%M:%S")}')
    print(f'{"="*60}')
    
    # ── Lire les donnees accumulees depuis HDFS ──
    try:
        trips = spark.read.parquet(HDFS_STREAMING_PATH)
        nb_rows = trips.count()
        print(f'Donnees accumulees dans HDFS : {nb_rows} lignes')
    except Exception as e:
        print(f'Pas encore de donnees dans HDFS : {e}')
        return
    
    if nb_rows == 0:
        print('Aucune donnee a traiter')
        return
    
    # ── Agregation 1 : Stats par borough (batch exact) ──
    stats_borough = (trips
        .join(broadcast(locations),
              trips.pickup_location_id == locations.location_id, 'left')
        .groupBy('borough')
        .agg(
            count('*').alias('nb_trips'),
            round(sum('total_amount'), 2).alias('total_revenue'),
            round(avg('trip_distance'), 2).alias('avg_distance'),
            round(avg('total_amount'), 2).alias('avg_fare')
        )
        .orderBy(desc('nb_trips'))
    )
    
    print('\n--- Stats Borough (batch reprocess) ---')
    stats_borough.show()
    
    pdf_borough = stats_borough.toPandas()
    pdf_borough = pdf_borough.dropna(subset=['borough'])
    ch_client.command('TRUNCATE TABLE reprocess_borough_stats')
    ch_client.insert_df('reprocess_borough_stats', pdf_borough)
    
    # ── Agregation 2 : Top zones (batch exact) ──
    top_zones = (trips
        .join(broadcast(locations),
              trips.pickup_location_id == locations.location_id, 'left')
        .groupBy('zone', 'borough')
        .agg(count('*').alias('nb_trips'))
        .orderBy(desc('nb_trips'))
        .limit(10)
    )
    
    print('--- Top 10 Zones (batch reprocess) ---')
    top_zones.show(truncate=False)
    
    pdf_zones = top_zones.toPandas()
    pdf_zones = pdf_zones.dropna(subset=['zone'])
    ch_client.command('TRUNCATE TABLE reprocess_top_zones')
    ch_client.insert_df('reprocess_top_zones', pdf_zones)
    
    # ── Agregation 3 : Stats par jour (batch exact) ──
    stats_daily = (trips
        .join(broadcast(dates), trips.date_id == dates.date_id, 'left')
        .groupBy('day')
        .agg(
            count('*').alias('nb_trips'),
            round(sum('total_amount'), 2).alias('total_revenue'),
            round(avg('total_amount'), 2).alias('avg_fare')
        )
        .orderBy('day')
    )
    
    print('--- Stats par Jour de Decembre (batch reprocess) ---')
    stats_daily.show()
    
    pdf_daily = stats_daily.toPandas()
    pdf_daily = pdf_daily.dropna(subset=['day'])
    ch_client.command('TRUNCATE TABLE reprocess_daily_stats')
    ch_client.insert_df('reprocess_daily_stats', pdf_daily)
    
    # ── Fusionner les resultats December dans les tables batch permanentes ──
    merge_reprocess_into_batch()
    
    # ── Absorb: move december data into silver (permanent batch) ──
    print('\n--- Absorption des donnees dans Silver ---')
    trips.write.mode('append').parquet(SILVER_TRIPS_PATH)
    print(f'  {nb_rows} lignes ajoutees a silver/trips_clean')
    
    # ── Stop streaming + producer ──
    print('\n--- Reset complet du streaming ---')
    import subprocess as _sp
    _sp.run(['pkill', '-f', 'streaming_all.py'], capture_output=True)
    _sp.run(['pkill', '-f', 'kafka_producer.py'], capture_output=True)
    print('  Streaming + Producer arretes')
    time.sleep(5)
    
    # ── Clear HDFS: december_trips + checkpoint ──
    hadoop_conf = spark._jsc.hadoopConfiguration()
    fs = spark._jvm.org.apache.hadoop.fs.FileSystem.get(hadoop_conf)
    for p in [HDFS_STREAMING_PATH, HDFS_CHECKPOINT_PATH]:
        hdfs_path = spark._jvm.org.apache.hadoop.fs.Path(p)
        if fs.exists(hdfs_path):
            fs.delete(hdfs_path, True)
            print(f'  HDFS supprime: {p}')
    
    # ── Purge Kafka topic (delete + recreate so streaming starts clean) ──
    try:
        from kafka.admin import KafkaAdminClient, NewTopic
        admin = KafkaAdminClient(bootstrap_servers=KAFKA_BROKER)
        admin.delete_topics([KAFKA_TOPIC])
        print(f'  Kafka topic {KAFKA_TOPIC} supprime')
        time.sleep(3)
        admin.create_topics([NewTopic(name=KAFKA_TOPIC, num_partitions=1, replication_factor=1)])
        print(f'  Kafka topic {KAFKA_TOPIC} recree (vide)')
        admin.close()
    except Exception as kafka_err:
        print(f'  Kafka reset: {kafka_err}')
    
    # ── Clear streaming tables in ClickHouse (fresh start) ──
    ch_client.command('TRUNCATE TABLE stream_borough_stats')
    ch_client.command('TRUNCATE TABLE stream_top_zones')
    ch_client.command('TRUNCATE TABLE stream_alerts')
    print('  Tables stream_* videes')
    
    # ── Restart streaming ──
    import sys as _sys
    _sp.Popen(
        [_sys.executable, '/home/jovyan/work/scripts/streaming_all.py'],
        stdout=open('/tmp/streaming_all.log', 'a'),
        stderr=_sp.STDOUT
    )
    print('  Streaming relance (clean slate)')
    
    # ── Log du cycle ──
    end = datetime.datetime.now()
    duration = (end - start).total_seconds()
    
    log_entry = {
        'cycle_num': cycle_num,
        'executed_at': end,
        'rows_processed': nb_rows,
        'duration_seconds': builtins.round(duration, 1),
        'boroughs_found': len(pdf_borough),
        'zones_found': len(pdf_zones)
    }
    import pandas as pd
    ch_client.insert_df('reprocess_log', pd.DataFrame([log_entry]))
    
    print(f'\nCycle {cycle_num} termine en {builtins.round(duration, 1)}s')
    print(f'  {nb_rows} lignes absorbees dans batch, streaming repart de zero')

In [ ]:
import pandas as pd

def merge_reprocess_into_batch():
    """Merge December reprocess results into the permanent batch tables (Jan-Nov + Dec)."""
    df_reprocess = ch_client.query_df('SELECT * FROM reprocess_borough_stats FINAL')
    if df_reprocess.empty:
        print('Pas de donnees reprocess a fusionner')
        return

    # ── borough stats: weighted merge of Jan-Nov + December ──
    df_batch = ch_client.query_df('SELECT * FROM batch_stats_borough FINAL')
    merged = df_batch.merge(df_reprocess, on='borough', how='outer', suffixes=('_b', '_r'))
    merged = merged.fillna(0)
    merged['nb_trips'] = merged['nb_trips_b'] + merged['nb_trips_r']
    merged['total_revenue'] = (merged['total_revenue_b'] + merged['total_revenue_r']).round(2)
    total = merged['nb_trips'].replace(0, 1)
    merged['avg_distance'] = ((merged['avg_distance_b'] * merged['nb_trips_b'] +
                               merged['avg_distance_r'] * merged['nb_trips_r']) / total).round(2)
    merged['avg_fare'] = ((merged['avg_fare_b'] * merged['nb_trips_b'] +
                           merged['avg_fare_r'] * merged['nb_trips_r']) / total).round(2)
    result_borough = merged[['borough', 'nb_trips', 'avg_distance', 'avg_fare', 'total_revenue']].copy()
    result_borough['nb_trips'] = result_borough['nb_trips'].astype('uint64')
    ch_client.insert_df('batch_stats_borough', result_borough)
    print(f'batch_stats_borough mis a jour : {len(result_borough)} boroughs (Jan-Dec)')

    # ── monthly stats: upsert December row ──
    df_daily = ch_client.query_df('SELECT * FROM reprocess_daily_stats FINAL')
    if not df_daily.empty:
        dec_nb_trips = int(df_daily['nb_trips'].sum())
        dec_revenue = builtins.round(float(df_daily['total_revenue'].sum()), 2)
        w = df_reprocess['nb_trips'].sum()
        dec_avg_dist = builtins.round(
            float((df_reprocess['avg_distance'] * df_reprocess['nb_trips']).sum() / w) if w > 0 else 0.0, 2
        )
        dec_row = pd.DataFrame([{
            'month': 12, 'year': 2023,
            'nb_trips': dec_nb_trips,
            'total_revenue': dec_revenue,
            'avg_distance': dec_avg_dist
        }])
        ch_client.insert_df('batch_stats_monthly', dec_row)
        print(f'batch_stats_monthly: Decembre insere ({dec_nb_trips:,} trajets)')

    # ── top zones: merge counts, re-rank top 10 ──
    df_batch_zones = ch_client.query_df('SELECT * FROM batch_top_zones FINAL')
    df_reprocess_zones = ch_client.query_df('SELECT * FROM reprocess_top_zones FINAL')
    if not df_reprocess_zones.empty:
        combined = pd.concat([df_batch_zones, df_reprocess_zones])
        merged_zones = (combined.groupby(['zone', 'borough'], as_index=False)['nb_trips']
                        .sum().sort_values('nb_trips', ascending=False).head(10))
        merged_zones['nb_trips'] = merged_zones['nb_trips'].astype('uint64')
        ch_client.insert_df('batch_top_zones', merged_zones)
        print(f'batch_top_zones mis a jour : top 10 fusionnes (Jan-Dec)')

    print('Fusion reprocess → batch terminee')

print('Fonction merge_reprocess_into_batch definie')

In [4]:
# ══════════════════════════════════════════════════
# BOUCLE DE REPROCESSING PERIODIQUE
# ══════════════════════════════════════════════════
# Lance un batch toutes les BATCH_INTERVAL_SECONDS
# A chaque cycle, les donnees accumulees dans HDFS
# grandissent (le streaming continue d'ecrire)
# → le batch reprocesse TOUT depuis le debut

print(f'Demarrage du batch reprocessing periodique')
print(f'Intervalle : toutes les {BATCH_INTERVAL_SECONDS} secondes')
print(f'Cycles : {NB_CYCLES}')
print(f'\nLancez le producer (notebook 03) et le consumer HDFS (notebook 05)')
print(f'en parallele pour voir les donnees augmenter a chaque cycle.\n')

cycle = 1
while NB_CYCLES == -1 or cycle <= NB_CYCLES:
    run_batch_reprocessing(cycle)
    
    if NB_CYCLES != -1 and cycle >= NB_CYCLES:
        break
    
    print(f'\nProchain batch dans {BATCH_INTERVAL_SECONDS} secondes...')
    time.sleep(BATCH_INTERVAL_SECONDS)
    cycle += 1

print(f'\n{"="*60}')
print(f'BATCH REPROCESSING TERMINE - {cycle} cycles executes')
print(f'{"="*60}')

Demarrage du batch reprocessing periodique
Intervalle : toutes les 30 secondes
Cycles : 5

Lancez le producer (notebook 03) et le consumer HDFS (notebook 05)
en parallele pour voir les donnees augmenter a chaque cycle.


CYCLE 1 - 02:50:00
Donnees accumulees dans HDFS : 53675 lignes

--- Stats Borough (batch reprocess) ---
+-------------+--------+-------------+------------+--------+
|      borough|nb_trips|total_revenue|avg_distance|avg_fare|
+-------------+--------+-------------+------------+--------+
|    Manhattan|   26671|    672675.17|        2.41|   25.22|
|        Bronx|   11885|    380067.54|        3.34|   31.98|
|       Queens|    8821|     357223.1|        5.97|    40.5|
|     Brooklyn|    5678|    154432.26|        2.89|    27.2|
|Staten Island|     603|     17859.43|        3.14|   29.62|
|          EWR|      17|      1669.88|        0.01|   98.23|
+-------------+--------+-------------+------------+--------+



DatabaseError: Received ClickHouse exception, code: 60, server response: Code: 60. DB::Exception: Table default.reprocess_borough_stats does not exist. Maybe you meant stream_borough_stats?. (UNKNOWN_TABLE) (version 23.8.16.16 (official build)) (for url http://clickhouse:8123)

In [ ]:
# ══════════════════════════════════════════════════
# COMPARAISON : Streaming vs Batch Reprocess
# ══════════════════════════════════════════════════
# Montre la difference entre les resultats streaming
# (approximatifs, incrementaux) et les resultats batch
# (exacts, recalcules de zero)

print('=== STREAMING (temps reel, approximatif) ===')
df_stream = ch_client.query_df('SELECT * FROM stream_borough_stats FINAL ORDER BY nb_trips DESC')
print(df_stream)

print('\n=== BATCH REPROCESS (periodique, exact) ===')
df_batch = ch_client.query_df('SELECT * FROM reprocess_borough_stats ORDER BY nb_trips DESC')
print(df_batch)

print('\n=== HISTORIQUE DES CYCLES ===')
df_log = ch_client.query_df('SELECT * FROM reprocess_log ORDER BY cycle_num')
print(df_log)

In [ ]:
ch_client.close()
spark.stop()
print('Connexions fermees')